In [16]:
import datasets

In [ ]:
ds = datasets.load_dataset("internlm/SWE-Fixer-Train-Editing-CoT-70K", split="train")
ds

In [21]:
prompt_template = """\
In this task, you will be provided with a software development issue from a real-world GitHub repository, along with the full content of relevant code files for modification. Your objective is to carefully analyze and understand the issue in the context of the provided files and identify the exact file paths and original code snippets that require modification. Based on this analysis, you will propose new code snippets to replace the identified ones to effectively resolve the issue.
After you're done thinking, recite the file paths and exact lines of code you want to change with their line numbers and then propose your edit.

After you're done thinking, answer in JSON format like this:
```json
[
    {{
        "file": "some/file/path.py",
        "code snippet to be_modified": "123     def some_function():\\n124         return False",
        "edited code snippet": "    def some_function():\\n        return True"
    }},
    {{
        "file": "some/other/file/path/to/some/file.py", 
        "code snippet to be_modified": "45 def validate_input(user_data):\\n46     if not isinstance(user_data, dict):\\n47         return None",
        "edited code snippet": "def validate_input(user_data):\\n    if not isinstance(user_data, dict):\\n        raise ValueError(\"Input must be a dictionary\")"
    }}
]
```


# Issue description
{issue_description}

# Relevant code files
{files}
"""

In [19]:
ds_debug = ds.select(range(100))

In [ ]:
ds_up = ds_debug.map(lambda x, idx: {"problem_id": f"swe_fixer_{idx}"}, with_indices=True)
ds_up = ds_up.map(lambda x: {"source": "internlm/SWE-Fixer-Train-Editing-CoT-70K", "task_type": "swe_fixer"})

# Function to format code files for display
def format_files(files):
    formatted = ""
    for file_info in files:
        formatted += f"## `{file_info['file']}`\n```\n{file_info['file content']}\n```\n\n"
    return formatted

ds_up = ds_up.map(lambda x: {"in_source_id": x["instance_id"]})

# Format the prompt using the template
ds_up = ds_up.map(lambda x: {
    "prompt": prompt_template.format(
        issue_description=x['input']['input']['issue'],
        files=format_files(x['input']['input']['files to be modified'])
    )
})

# Format the golden_standard_solution properly - use repr() to ensure it's a valid Python literal
ds_up = ds_up.map(lambda x: {"golden_standard_solution": repr({
    "edited code": x["output"]["edited code"]
})})

ds_up = ds_up.map(lambda x: {"verification_info": repr({
    "input": x["input"]["input"]
})})

# Format the metadata as a string representation of a dictionary
ds_up = ds_up.map(lambda x: {"metadata": repr({
    # "input": x["input"]["input"]
})})

ds_up = ds_up.select_columns(["problem_id", "source", "task_type", "in_source_id", "prompt", "golden_standard_solution", "verification_info", "metadata"])

In [ ]:
# Preview the formatted prompt for the first example
print("FORMATTED PROMPT PREVIEW:\n")
print(ds_up[0]["prompt"]) # Show the first 2000 characters for preview

# Test if the serialized data can be correctly parsed
import ast

# Test with the first example
example = ds_up[0]
print("Testing verification_info...")
try:
    parsed_verification_info = ast.literal_eval(example["verification_info"])
    print("✅ Successfully parsed verification_info")
    print(f"Keys: {list(parsed_verification_info.keys())}")
except Exception as e:
    print(f"❌ Error parsing verification_info: {e}")
    
print("\nTesting metadata...")
try:
    parsed_metadata = ast.literal_eval(example["metadata"])
    print("✅ Successfully parsed metadata")
    print(f"Keys: {list(parsed_metadata.keys())}")
except Exception as e:
    print(f"❌ Error parsing metadata: {e}")

# If everything is good, save to JSONL for verification
import json
import os

output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/swe_fixer_examples.jsonl"

with open(output_file, "w") as f:
    for item in ds_up:
        # Convert to dict for serialization
        item_dict = {k: v for k, v in item.items()}
        dat = json.dumps(item_dict)
        # f.write(dat + "\n")
        
# print(f"\nSaved {len(ds_up)} examples to {output_file}")
print(f"To verify: python -m genesys.verify --file {output_file}")

In [ ]:
# Push the fixed dataset to HuggingFace Hub
ds_up.push_to_hub("rasdani/swe-fixer-debug-pi-format")